# EU G4 Debt-at-Risk — End-to-End Pipeline

This notebook walks through the complete **Debt-at-Risk (DaR)** methodology
from IMF Working Paper WP/25/86 (Furceri, Giannone, Kisat, Lam, Li — May 2025),
applied to the EU G4 (France, Germany, Italy, Spain).

## Phases
1. **Data pipeline** — pull WEO, FSI, ECB spreads, WUI
2. **Location-scale quantile regression** (MSS 2019)
3. **Density fitting + pooling** (skewed-t + log-score)
4. **Debt-at-Risk extraction** (P5/P50/P95)
5. **Fiscal crisis signal** (panel logit)
6. **Charts + PowerPoint deck**

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

# Ensure project root is on PYTHONPATH
project_root = Path('.').resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
%matplotlib inline

print('Setup complete.')

---
## Phase 1 — Data Pipeline

In [ ]:
from data.imf_weo import fetch_weo

print('Fetching IMF WEO data (this may take ~30s)...')
weo = fetch_weo(save=True)
print(f'WEO panel shape: {weo.shape}')
weo.head()

In [ ]:
from data.imf_fsi import fetch_fsi

print('Fetching IMF FSI data...')
fsi = fetch_fsi(save=True)
print(f'FSI panel shape: {fsi.shape}')
fsi.head()

In [ ]:
from data.ecb_spreads import fetch_ecb_spreads

print('Fetching ECB sovereign spreads...')
ecb = fetch_ecb_spreads(save=True)
print(f'ECB spreads shape: {ecb.shape}')
ecb.head()

In [ ]:
from data.wui import fetch_wui

print('Loading WUI data (will attempt download if not cached)...')
wui = fetch_wui(save=True)
print(f'WUI panel shape: {wui.shape}')
wui.head()

In [ ]:
from data.pipeline import build_panel

print('Building merged estimation panel...')
panel = build_panel(save=True)
print(f'Final panel shape: {panel.shape}')
print(f'Countries: {panel["iso"].nunique()}')
print(f'Years: {panel["year"].min()} – {panel["year"].max()}')
panel.describe()

In [ ]:
# Quick visualisation — G4 historical debt
G4 = {'FRA': 'France', 'DEU': 'Germany', 'ITA': 'Italy', 'ESP': 'Spain'}
G4_COLORS = {'FRA': '#2980B9', 'DEU': '#27AE60', 'ITA': '#E74C3C', 'ESP': '#F39C12'}

fig, ax = plt.subplots(figsize=(10, 5))
for iso, label in G4.items():
    g = panel[(panel['iso'] == iso) & (panel['year'] >= 1995)]
    ax.plot(g['year'], g['debt_gdp'], label=label, color=G4_COLORS[iso], linewidth=2)
ax.set_title('EU G4 — Government Debt / GDP (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Debt / GDP (%)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Phase 2 — Location-Scale Quantile Regression

In [ ]:
from model.location_scale import run_all

print('Running MSS (2019) three-step estimator for all conditioning variables × horizons...')
print('(This may take several minutes for the full panel)')
quantile_preds = run_all(panel, save=True)
print(f'\nQuantile predictions shape: {quantile_preds.shape}')
quantile_preds.head()

In [ ]:
# Fit skewed-t densities
from model.quantile_fit import fit_densities

print('Fitting Azzalini-Capitanio skewed-t distributions...')
density_params = fit_densities(quantile_preds, save=True)
print(f'Density parameters shape: {density_params.shape}')
density_params.head()

---
## Phase 3 — Density Pooling & Debt-at-Risk

In [ ]:
from risk.pooling import compute_pooled_weights, build_pooled_density

print('Computing log-score pooling weights...')
weights = compute_pooled_weights(density_params, save=True)
print(f'Weights shape: {weights.shape}')
weights.head()

In [ ]:
print('Building pooled density...')
pooled = build_pooled_density(density_params, weights, save=True)
print(f'Pooled density shape: {pooled.shape}')
pooled.head()

In [ ]:
from risk.dar import extract_dar, get_g4_dar

print('Extracting Debt-at-Risk (h=3, re-centred to WEO April 2025 baselines)...')
dar = extract_dar(pooled, save=True)
print('\nG4 DaR summary:')
g4_dar = dar[dar['iso'].isin(G4.keys())]
display(g4_dar[['iso', 'Q05', 'Q50', 'Q50_weo', 'Q95', 'DaR_weo', 'Upside_weo', 'Downside']].to_string(index=False))

---
## Phase 4 — Fiscal Crisis Signal

In [ ]:
from crisis.logit_signal import run_crisis_logit

print('Running fiscal crisis logit model...')
crisis_scores = run_crisis_logit(quantile_preds, save=True)
print(f'Crisis scores shape: {crisis_scores.shape}')

g4_crisis = crisis_scores[
    crisis_scores['iso'].isin(G4.keys()) &
    (crisis_scores['year'] == crisis_scores['year'].max())
].drop_duplicates('iso')[['iso', 'crisis_prob_avg']]

print('\nG4 Fiscal Crisis Probability Scores:')
print(g4_crisis.to_string(index=False))

---
## Phase 5 — Charts

In [ ]:
from output.charts import plot_fan_charts, plot_dar_comparison, plot_waterfall, plot_crisis_signals

WEO_BASELINES = {'FRA': 117.0, 'DEU': 65.0, 'ITA': 135.0, 'ESP': 109.0}

print('Building fan charts...')
fig_fan = plot_fan_charts(panel, pooled, weo_baselines=WEO_BASELINES, save=True)
plt.show()

In [ ]:
print('Building DaR comparison chart...')
fig_dar = plot_dar_comparison(dar, weo_baselines=WEO_BASELINES, save=True)
plt.show()

In [ ]:
print('Building waterfall charts (risk decomposition)...')
for iso in G4:
    fig_wf = plot_waterfall(quantile_preds, iso, save=True)
    plt.show()

In [ ]:
print('Building crisis signal chart...')
latest_year = crisis_scores['year'].max() if not crisis_scores.empty else 2024
fig_crisis = plot_crisis_signals(crisis_scores, year=latest_year, save=True)
plt.show()

---
## Phase 5 — PowerPoint Deck

In [ ]:
from output.deck import build_deck

print('Building PowerPoint deck...')
prs = build_deck(panel=panel, save=True)
print('Done! Deck saved to output/eu_g4_debt_at_risk.pptx')

---
## Summary

The pipeline has:
1. ✅ Pulled WEO, FSI, ECB spreads, and WUI data
2. ✅ Built an estimation panel of ~40–60 countries (1990–2024)
3. ✅ Estimated location-scale quantile regressions (MSS 2019) for 7 conditioning variables × 3 horizons
4. ✅ Fitted Azzalini-Capitanio skewed-t densities to quantile predictions
5. ✅ Combined densities using log-score pooling (Crump et al. 2023)
6. ✅ Extracted Debt-at-Risk (P5/P50/P95) re-centred to WEO April 2025 baselines
7. ✅ Produced fiscal crisis probability scores via panel logit
8. ✅ Generated fan charts, waterfall charts, and comparison charts
9. ✅ Produced a 5-slide board-ready PowerPoint deck

**Output**: `output/eu_g4_debt_at_risk.pptx`